In [5]:
# -*- coding: utf-8 -*-
"""
Tarea 3 - Aprendizaje por Refuerzo
Asignatura: Ingeniería de Software · 7mo Semestre
Estudiante: Isaac Torres

Este script contiene la implementación completa y unificada de la tarea, dividida en tres
secciones claramente separadas (Parte A, Parte B y Parte C) y configurada para generar y guardar
los tres gráficos requeridos en el directorio raíz.
"""

import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import os
# Primera celda de configuración en Google Colab
!pip install gymnasium pygame > /dev/null 2>&1
print("✅ Entorno de Gymnasium y Pygame listos para CartPole, MountainCar y EcomInventoryEnv.")


✅ Entorno de Gymnasium y Pygame listos para CartPole, MountainCar y EcomInventoryEnv.


==============================================================================
PARTE A — Diagnóstico del Agente Roto (CartPole-v1)


==============================================================================


In [6]:
print("=" * 80)
print("EJECUTANDO: PARTE A — DIAGNÓSTICO Y CORRECCIÓN (CartPole-v1)")
print("=" * 80)

# Diagnóstico de los 4 errores conceptuales identificados:
# 1. GAMMA = 0.0 (Hace al agente miope, ignorando retornos futuros. Corrección: GAMMA = 0.99).
# 2. EPS_DECAY = 1.0 (Evita que epsilon decaiga, manteniendo exploración aleatoria permanente. Corrección: EPS_DECAY = 0.995).
# 3. a = np.argmax(Q[s]) (Explota desde el inicio sin explorar previamente. Corrección: Implementar Epsilon-Greedy).
# 4. Q[s][a] = ALPHA * (...) (Sobrescribe Q ignorando el valor anterior en vez de acumular. Corrección: Usar +=).

def entrenar_agente_roto():
    print("\nEntrenando Agente Roto en CartPole-v1...")
    env = gym.make('CartPole-v1')
    N_BINS = 10
    limites = [(-2.4, 2.4), (-3.0, 3.0), (-0.3, 0.3), (-3.0, 3.0)]
    bins = [np.linspace(l, h, N_BINS-1) for l, h in limites]

    def disc(obs):
        return tuple(int(np.digitize(obs[i], bins[i])) for i in range(4))

    Q = np.zeros([N_BINS]*4+[2])
    ALPHA = 0.1; GAMMA = 0.0; EPSILON = 1.0; EPS_MIN = 0.01; EPS_DECAY = 1.0
    rewards = []

    for ep in range(3000):
        obs, _ = env.reset(seed=ep)
        s = disc(obs)
        total = 0
        done = False
        while not done:
            a = np.argmax(Q[s])  # Error 3: Siempre explotar (sin epsilon-greedy)
            obs2, r, done, trunc, _ = env.step(a)
            s2 = disc(obs2)
            done = done or trunc
            # Error 4: Asignación directa en vez de actualización incremental (+=)
            # Error 1: GAMMA = 0.0
            Q[s][a] = ALPHA * (r + GAMMA * np.max(Q[s2]) - Q[s][a])
            s = s2
            total += r
        rewards.append(total)
    env.close()
    return rewards

def entrenar_agente_corregido():
    print("Entrenando Agente Corregido en CartPole-v1...")
    env = gym.make('CartPole-v1')
    N_BINS = 10
    limites = [(-2.4, 2.4), (-3.0, 3.0), (-0.3, 0.3), (-3.0, 3.0)]
    bins = [np.linspace(l, h, N_BINS-1) for l, h in limites]

    def disc(obs):
        return tuple(int(np.digitize(obs[i], bins[i])) for i in range(4))

    Q = np.zeros([N_BINS]*4+[2])
    # Corrección 1 (GAMMA = 0.99) y Corrección 2 (EPS_DECAY = 0.995)
    ALPHA = 0.1; GAMMA = 0.99; EPSILON = 1.0; EPS_MIN = 0.01; EPS_DECAY = 0.995
    rewards = []

    for ep in range(3000):
        obs, _ = env.reset(seed=ep)
        s = disc(obs)
        total = 0
        done = False
        while not done:
            # Corrección 3: Implementación de política de exploración Epsilon-Greedy
            if np.random.random() < EPSILON:
                a = env.action_space.sample()
            else:
                a = np.argmax(Q[s])

            obs2, r, done, trunc, _ = env.step(a)
            s2 = disc(obs2)
            done = done or trunc

            # Corrección 4: Actualización incremental correcta usando +=
            Q[s][a] += ALPHA * (r + GAMMA * np.max(Q[s2]) - Q[s][a])
            s = s2
            total += r

        # Decaimiento del epsilon
        if EPSILON > EPS_MIN:
            EPSILON *= EPS_DECAY

        rewards.append(total)
    env.close()
    return rewards, Q

# Ejecutar y graficar Parte A
rewards_roto = entrenar_agente_roto()
rewards_corregido, Q_corregido = entrenar_agente_corregido()

def media_movil(data, window=100):
    ret = np.cumsum(data, dtype=float)
    ret[window:] = ret[window:] - ret[:-window]
    return ret[window - 1:] / window

ma_roto = media_movil(rewards_roto)
ma_corregido = media_movil(rewards_corregido)

# Gráfico 1: Roto vs Corregido (CartPole-v1)
plt.figure(figsize=(10, 6))
plt.plot(ma_roto, label="Agente Roto (Con 4 errores)", color="red", alpha=0.8)
plt.plot(ma_corregido, label="Agente Corregido (Parámetros y actualización corregida)", color="blue", alpha=0.8)
plt.title("Parte A: Curva de Aprendizaje CartPole-v1 (Media Móvil de 100 Episodios)")
plt.xlabel("Episodios")
plt.ylabel("Recompensa Acumulada Promedio")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig("curva_aprendizaje_cartpole.png", dpi=150, bbox_inches="tight")
print("-> Gráfico 'curva_aprendizaje_cartpole.png' guardado exitosamente en el directorio raíz.")
plt.close()




EJECUTANDO: PARTE A — DIAGNÓSTICO Y CORRECCIÓN (CartPole-v1)

Entrenando Agente Roto en CartPole-v1...
Entrenando Agente Corregido en CartPole-v1...
-> Gráfico 'curva_aprendizaje_cartpole.png' guardado exitosamente en el directorio raíz.


==============================================================================
PARTE B — Nuevo Entorno: MountainCar-v0


==============================================================================


In [7]:
print("\n" + "=" * 80)
print("EJECUTANDO: PARTE B — ENTORNO MountainCar-v0")
print("=" * 80)

def entrenar_mountaincar():
    print("Entrenando Q-Learning en MountainCar-v0 (5,000 episodios)...")
    env = gym.make('MountainCar-v0')

    # Justificación de bins (20): Ofrece un buen balance entre granularidad y tamaño de la Q-tabla (400 estados).
    N_BINS_POS = 20
    N_BINS_VEL = 20

    pos_bins = np.linspace(-1.2, 0.6, N_BINS_POS - 1)
    vel_bins = np.linspace(-0.07, 0.07, N_BINS_VEL - 1)

    def disc(obs):
        pos_idx = int(np.digitize(obs[0], pos_bins))
        vel_idx = int(np.digitize(obs[1], vel_bins))
        return (pos_idx, vel_idx)

    Q = np.zeros((N_BINS_POS, N_BINS_VEL, 3))

    # Hiperparámetros justificados para MountainCar:
    # GAMMA = 0.99 para valorar a largo plazo debido a que la recompensa es siempre -1.
    # EPS_DECAY = 0.9992 (decaimiento lento) para permitir explorar lo suficiente al inicio.
    ALPHA = 0.1; GAMMA = 0.99; EPSILON = 1.0; EPS_MIN = 0.01; EPS_DECAY = 0.9992
    rewards = []

    for ep in range(5000):
        obs, _ = env.reset(seed=ep)
        s = disc(obs)
        total = 0
        done = False
        while not done:
            if np.random.random() < EPSILON:
                a = env.action_space.sample()
            else:
                a = np.argmax(Q[s])

            obs2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            s2 = disc(obs2)

            Q[s][a] += ALPHA * (r + GAMMA * np.max(Q[s2]) - Q[s][a])
            s = s2
            total += r

        if EPSILON > EPS_MIN:
            EPSILON *= EPS_DECAY
        rewards.append(total)
    env.close()
    return rewards

rewards_mountaincar = entrenar_mountaincar()
ma_mountaincar = media_movil(rewards_mountaincar)

# Gráfico 2: Curva MountainCar-v0
plt.figure(figsize=(10, 6))
plt.plot(rewards_mountaincar, color="lightblue", alpha=0.5, label="Recompensa por Episodio")
plt.plot(np.arange(99, 5000), ma_mountaincar, color="navy", label="Media Móvil (100 episodios)")
plt.title("Parte B: Curva de Aprendizaje en MountainCar-v0 (Q-Learning)")
plt.xlabel("Episodios")
plt.ylabel("Recompensa Acumulada")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig("curva_aprendizaje_mountaincar.png", dpi=150, bbox_inches="tight")
print("-> Gráfico 'curva_aprendizaje_mountaincar.png' guardado exitosamente en el directorio raíz.")
plt.close()





EJECUTANDO: PARTE B — ENTORNO MountainCar-v0
Entrenando Q-Learning en MountainCar-v0 (5,000 episodios)...
-> Gráfico 'curva_aprendizaje_mountaincar.png' guardado exitosamente en el directorio raíz.


==============================================================================
PARTE C — Diseño de Entorno Propio (Simulado - Agente de Inventario)


==============================================================================


In [8]:
print("\n" + "=" * 80)
print("EJECUTANDO: PARTE C — DISEÑO DE ENTORNO PROPIO (Simulador de Inventario)")
print("=" * 80)

class InventoryEnv:
    """
    Entorno simulado en Python puro para la gestión de inventario.
    El estado representa las unidades en stock al inicio de cada día.
    Las acciones corresponden a las unidades a ordenar al proveedor.
    """
    def __init__(self, max_stock=20, lambda_demand=5, price=10, order_cost=3, holding_cost=1, stockout_penalty=5, max_steps=30):
        self.max_stock = max_stock
        self.lambda_demand = lambda_demand
        self.price = price
        self.order_cost = order_cost
        self.holding_cost = holding_cost
        self.stockout_penalty = stockout_penalty
        self.max_steps = max_steps
        self.state = 0
        self.steps = 0
        self.action_to_qty = {0: 0, 1: 5, 2: 10, 3: 15, 4: 20}
        self.action_space_size = len(self.action_to_qty)
        self.state_space_size = self.max_stock + 1

    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.state = 10  # Stock inicial de 10 unidades
        self.steps = 0
        return self.state

    def step(self, action):
        qty_ordered = self.action_to_qty[action]
        stock_before = min(self.state + qty_ordered, self.max_stock)
        demand = np.random.poisson(self.lambda_demand)

        units_sold = min(stock_before, demand)
        unsatisfied_demand = max(0, demand - stock_before)

        self.state = max(0, stock_before - demand)

        # Ingresos - Costos (pedido, almacenamiento y penalización de quiebre)
        revenue = units_sold * self.price
        purchase_cost = qty_ordered * self.order_cost
        holding_cost_val = self.state * self.holding_cost
        penalty = unsatisfied_demand * self.stockout_penalty

        reward = revenue - purchase_cost - holding_cost_val - penalty
        self.steps += 1
        done = self.steps >= self.max_steps

        return self.state, reward, done

def entrenar_entorno_propio():
    print("Entrenando Q-Learning en Entorno de Inventario Propio (5,000 episodios)...")
    env = InventoryEnv()
    Q = np.zeros((env.state_space_size, env.action_space_size))

    ALPHA = 0.1; GAMMA = 0.9; EPSILON = 1.0; EPS_MIN = 0.01; EPS_DECAY = 0.998
    rewards = []

    for ep in range(5000):
        s = env.reset(seed=ep)
        total = 0
        done = False
        while not done:
            if np.random.random() < EPSILON:
                a = np.random.choice(env.action_space_size)
            else:
                a = np.argmax(Q[s])

            s2, r, done = env.step(a)
            Q[s][a] += ALPHA * (r + GAMMA * np.max(Q[s2]) - Q[s][a])
            s = s2
            total += r

        if EPSILON > EPS_MIN:
            EPSILON *= EPS_DECAY
        rewards.append(total)
    return rewards

rewards_propio = entrenar_entorno_propio()
ma_propio = media_movil(rewards_propio)

# Gráfico 3: Curva Entorno Propio
plt.figure(figsize=(10, 6))
plt.plot(rewards_propio, color="lightgreen", alpha=0.4, label="Recompensa por Episodio")
plt.plot(np.arange(99, 5000), ma_propio, color="darkgreen", label="Media Móvil (100 episodios)")
plt.title("Parte C: Curva de Aprendizaje en Entorno de Inventario Propio (Q-Learning)")
plt.xlabel("Episodios")
plt.ylabel("Recompensa Acumulada")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig("curva_aprendizaje_entorno_propio.png", dpi=150, bbox_inches="tight")
print("-> Gráfico 'curva_aprendizaje_entorno_propio.png' guardado exitosamente en el directorio raíz.")
plt.close()

print("\n" + "=" * 80)
print("PROCESO DE VALIDACIÓN DE REQUISITOS PRE-SUBIDA FINALIZADO CON ÉXITO")
print("=" * 80)



EJECUTANDO: PARTE C — DISEÑO DE ENTORNO PROPIO (Simulador de Inventario)
Entrenando Q-Learning en Entorno de Inventario Propio (5,000 episodios)...
-> Gráfico 'curva_aprendizaje_entorno_propio.png' guardado exitosamente en el directorio raíz.

PROCESO DE VALIDACIÓN DE REQUISITOS PRE-SUBIDA FINALIZADO CON ÉXITO
